## Visualising GWAS results

This .ipynb file is used to take a .regenie file produced from regenie steps 1-3, visualise results as a Manhattan plot, and search for specific variants

Functions: set up Manhattan plot and variant lookup functions here

In [ ]:
## install packages if required

required_packages <- c("data.table", "ggplot2", "ggrepel", "knitr")

for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
  }
}


########################################
#===== Function 1: Manhattan plot =====#
########################################

plot_manhattan <- function(
    phenotype_name,
    gwas_directory,
    genetic_similarity_groups = c("eur", "afr", "eas", "cas", "sas", "mid", "amr"),
    gws_threshold = 5e-8,
    maximum_labels = 30L,
    label_window_bp = 500000L,
    label_mode = c("locus", "variant"),
    save_plots = TRUE,
    output_directory = getwd(),
    plot_width = 16,
    plot_height = 9,
    plot_dpi = 300,
    keep_downloaded_files = FALSE,
    overwrite_downloads = TRUE
) {

  # check arguments

  label_mode <- match.arg(label_mode)

  if (
    length(phenotype_name) != 1L ||
    is.na(phenotype_name) ||
    !nzchar(phenotype_name)
  ) {
    stop(
      "`phenotype_name` must be a single non-empty character string."
    )
  }

  if (
    length(gwas_directory) != 1L ||
    is.na(gwas_directory) ||
    !nzchar(gwas_directory)
  ) {
    stop(
      "`gwas_directory` must be a single non-empty character string."
    )
  }

  genetic_similarity_groups <- unique(
    toupper(genetic_similarity_groups)
  )

  if (length(genetic_similarity_groups) == 0L) {
    stop(
      "At least one genetic similarity group must be supplied."
    )
  }

  if (
    length(gws_threshold) != 1L ||
    is.na(gws_threshold) ||
    gws_threshold <= 0 ||
    gws_threshold >= 1
  ) {
    stop(
      "`gws_threshold` must be a number between 0 and 1."
    )
  }

  maximum_labels <- as.integer(maximum_labels)

  if (
    length(maximum_labels) != 1L ||
    is.na(maximum_labels) ||
    maximum_labels < 0L
  ) {
    stop(
      "`maximum_labels` must be a non-negative integer."
    )
  }

  label_window_bp <- as.numeric(label_window_bp)

  if (
    length(label_window_bp) != 1L ||
    is.na(label_window_bp) ||
    label_window_bp <= 0
  ) {
    stop(
      "`label_window_bp` must be a positive number."
    )
  }

  if (!nzchar(Sys.which("dx"))) {
    stop(
      paste0(
        "The DNAnexus `dx` command-line tool could not be found. ",
        "This function must be run in an environment where `dx` ",
        "is available and authenticated."
      )
    )
  }

  if (isTRUE(save_plots) && !dir.exists(output_directory)) {
    dir.create(
      output_directory,
      recursive = TRUE,
      showWarnings = FALSE
    )
  }


  # Helper function for running dx commands

  # Helper function for running dx commands through Bash
  run_dx <- function(args, capture_output = TRUE) {

    stderr_file <- tempfile(fileext = ".txt")

    on.exit(
      unlink(stderr_file),
      add = TRUE
    )

    quoted_args <- vapply(
      args,
      FUN = shQuote,
      FUN.VALUE = character(1L),
      type = "sh"
    )

    dx_command <- paste(
      "dx",
      paste(quoted_args, collapse = " ")
    )

    shell_command <- paste(
      "unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy;",
      dx_command
    )

    full_command <- paste(
      "bash -c",
      shQuote(shell_command, type = "sh"),
      "2>",
      shQuote(stderr_file, type = "sh")
    )

    output <- tryCatch(
      {
        if (isTRUE(capture_output)) {
          system(
            full_command,
            intern = TRUE
          )
        } else {
          status <- system(
            full_command,
            intern = FALSE
          )

          structure(
            character(),
            status = status
          )
        }
      },
      warning = function(w) {
        warning_status <- attr(
          conditionCall(w),
          "status"
        )

        result <- character()

        attr(result, "status") <- if (
          is.null(warning_status)
        ) {
          1L
        } else {
          warning_status
        }

        result
      },
      error = function(e) {
        result <- character()
        attr(result, "status") <- 1L
        attr(result, "error_message") <- conditionMessage(e)
        result
      }
    )

    status <- attr(output, "status")

    if (is.null(status)) {
      status <- 0L
    }

    stderr_output <- if (file.exists(stderr_file)) {
      readLines(
        stderr_file,
        warn = FALSE
      )
    } else {
      character()
    }

    list(
      output = as.character(output),
      error = stderr_output,
      status = as.integer(status),
      command = full_command
    )
  }


  # Search the specified DNAnexus directory

  message(
    "Searching DNAnexus directory: ",
    gwas_directory
  )

  directory_listing <- run_dx(
    args = c(
      "ls",
      gwas_directory
    )
  )

  if (directory_listing$status != 0L) {
    stop(
      paste0(
        "Could not list the supplied DNAnexus directory.\n",
        "Directory: ",
        gwas_directory,
        "\n",
        "Command attempted:\n",
        directory_listing$command,
        "\n",
        "DNAnexus/Bash output:\n",
        paste(directory_listing$error, collapse = "\n"),
        "\n",
        "Exit status: ",
        directory_listing$status
      )
    )
  }

  available_files <- trimws(directory_listing$output)
  available_files <- available_files[nzchar(available_files)]

  if (length(available_files) == 0L) {
    stop(
      paste0(
        "No files were found in: ",
        gwas_directory
      )
    )
  }

  message(
    "Found ",
    length(available_files),
    " entries in the supplied directory."
  )

  # Identify the requested REGENIE files

  file_information <- data.table::data.table(
    group = genetic_similarity_groups,
    expected_file = paste0(
      phenotype_name,
      "_",
      genetic_similarity_groups,
      ".regenie"
    )
  )

  file_information[
    ,
    matched_file := vapply(
      expected_file,
      FUN = function(expected_name) {

        matches <- available_files[
          tolower(basename(available_files)) ==
            tolower(expected_name)
        ]

        if (length(matches) == 0L) {
          return(NA_character_)
        }

        if (length(matches) > 1L) {
          warning(
            paste0(
              "Multiple directory entries matched ",
              expected_name,
              ". The first match will be used: ",
              matches[1L]
            ),
            call. = FALSE
          )
        }

        matches[1L]
      },
      FUN.VALUE = character(1L)
    )
  ]

  missing_groups <- file_information[
    is.na(matched_file),
    group
  ]

  if (length(missing_groups) > 0L) {
    warning(
      paste0(
        "No matching REGENIE file was found for: ",
        paste(missing_groups, collapse = ", "),
        ". These groups will be skipped."
      ),
      call. = FALSE
    )
  }

  file_information <- file_information[
    !is.na(matched_file)
  ]

  if (nrow(file_information) == 0L) {
    stop(
      paste0(
        "None of the requested REGENIE files were found. ",
        "Expected names such as: ",
        phenotype_name,
        "_EUR.regenie"
      )
    )
  }


  # Create a temporary download directory

  download_directory <- tempfile(
    pattern = "regenie_downloads_"
  )

  dir.create(
    download_directory,
    recursive = TRUE,
    showWarnings = FALSE
  )

  if (!isTRUE(keep_downloaded_files)) {
    on.exit(
      unlink(
        download_directory,
        recursive = TRUE,
        force = TRUE
      ),
      add = TRUE
    )
  }

  message(
    "Local download directory: ",
    download_directory
  )


  # Prepare output lists

  gwas_results <- list()
  significant_results <- list()
  labelled_results <- list()
  manhattan_plots <- list()
  download_paths <- list()


  # Standard chromosome order
  standard_chromosome_order <- c(
    as.character(1:22),
    "X",
    "Y",
    "XY",
    "MT"
  )

  log10p_threshold <- -log10(gws_threshold)


  # Process each genetic similarity group

  for (row_number in seq_len(nrow(file_information))) {

    group <- file_information$group[row_number]
    matched_file <- file_information$matched_file[row_number]

    local_file <- file.path(
      download_directory,
      basename(matched_file)
    )

    remote_file <- paste0(
      sub("/+$", "", gwas_directory),
      "/",
      matched_file
    )

    message("")
    message(
      "Processing group: ",
      group
    )

    message(
      "Remote file: ",
      remote_file
    )


    # Download the file

    download_arguments <- c(
      "download",
      remote_file,
      "-o",
      local_file
    )

    if (isTRUE(overwrite_downloads)) {
      download_arguments <- c(
        download_arguments,
        "--overwrite"
      )
    }

    download_result <- run_dx(
      args = download_arguments,
      capture_output = TRUE
    )

    if (
      download_result$status != 0L ||
      !file.exists(local_file)
    ) {
      warning(
        paste0(
          "The download failed for group ",
          group,
          ". This group will be skipped.\n",
          "Command attempted:\n",
          download_result$command,
          "\n",
          "DNAnexus/Bash output:\n",
          paste(download_result$error, collapse = "\n"),
          "\n",
          "Exit status: ",
          download_result$status
        ),
        call. = FALSE
      )

      next
    }

    download_paths[[group]] <- local_file

    message(
      "Successfully downloaded ",
      basename(local_file)
    )


    # Inspect the header before loading the full file

    header <- tryCatch(
      names(
        data.table::fread(
          local_file,
          nrows = 0L,
          showProgress = FALSE
        )
      ),
      error = function(e) {
        warning(
          paste0(
            "Could not read the header for group ",
            group,
            ": ",
            conditionMessage(e)
          ),
          call. = FALSE
        )

        NULL
      }
    )

    if (is.null(header)) {
      next
    }

    message(
      "Columns found for ",
      group,
      ": ",
      paste(header, collapse = ", ")
    )

    chromosome_column <- if ("CHROM" %in% header) {
      "CHROM"
    } else if ("#CHROM" %in% header) {
      "#CHROM"
    } else {
      warning(
        paste0(
          "Neither CHROM nor #CHROM was found for group ",
          group,
          ". This group will be skipped."
        ),
        call. = FALSE
      )

      next
    }

    required_columns <- c(
      chromosome_column,
      "GENPOS",
      "ID",
      "LOG10P"
    )

    missing_columns <- setdiff(
      required_columns,
      header
    )

    if (length(missing_columns) > 0L) {
      warning(
        paste0(
          "Group ",
          group,
          " is missing the following required columns: ",
          paste(missing_columns, collapse = ", "),
          ". This group will be skipped."
        ),
        call. = FALSE
      )

      next
    }


    # Load only the required columns

    gwas_data <- tryCatch(
      data.table::fread(
        local_file,
        select = required_columns,
        showProgress = TRUE
      ),
      error = function(e) {
        warning(
          paste0(
            "Could not import the GWAS results for group ",
            group,
            ": ",
            conditionMessage(e)
          ),
          call. = FALSE
        )

        NULL
      }
    )

    if (is.null(gwas_data)) {
      next
    }

    if (chromosome_column == "#CHROM") {
      data.table::setnames(
        gwas_data,
        old = "#CHROM",
        new = "CHROM"
      )
    }

    message(
      "Rows imported for ",
      group,
      ": ",
      format(
        nrow(gwas_data),
        big.mark = ","
      )
    )

    message(
      "Memory used by imported columns: ",
      format(
        object.size(gwas_data),
        units = "auto"
      )
    )


    # Clean the imported results

    gwas_data[
      ,
      CHROM := toupper(
        sub(
          "^CHR",
          "",
          trimws(as.character(CHROM)),
          ignore.case = TRUE
        )
      )
    ]

    gwas_data[
      CHROM %in% c("M", "MTDNA"),
      CHROM := "MT"
    ]

    gwas_data[
      ,
      GENPOS := suppressWarnings(
        as.numeric(GENPOS)
      )
    ]

    gwas_data[
      ,
      LOG10P := suppressWarnings(
        as.numeric(LOG10P)
      )
    ]

    gwas_data[
      ,
      ID := as.character(ID)
    ]

    number_before_cleaning <- nrow(gwas_data)

    gwas_data <- gwas_data[
      !is.na(CHROM) &
        !is.na(GENPOS) &
        !is.na(LOG10P) &
        GENPOS >= 0 &
        LOG10P >= 0
    ]

    number_removed <- number_before_cleaning - nrow(gwas_data)

    if (number_removed > 0L) {
      message(
        "Removed ",
        format(number_removed, big.mark = ","),
        " invalid or incomplete rows for ",
        group,
        "."
      )
    }

    chromosome_levels <- standard_chromosome_order[
      standard_chromosome_order %in% unique(gwas_data$CHROM)
    ]

    unexpected_chromosomes <- setdiff(
      unique(gwas_data$CHROM),
      standard_chromosome_order
    )

    if (length(unexpected_chromosomes) > 0L) {
      warning(
        paste0(
          "The following chromosome labels in ",
          group,
          " will not be plotted: ",
          paste(unexpected_chromosomes, collapse = ", ")
        ),
        call. = FALSE
      )
    }

    gwas_data <- gwas_data[
      CHROM %in% chromosome_levels
    ]

    if (nrow(gwas_data) == 0L) {
      warning(
        paste0(
          "No valid rows remained for group ",
          group,
          " after cleaning."
        ),
        call. = FALSE
      )

      next
    }

    gwas_data[
      ,
      CHROM := factor(
        CHROM,
        levels = chromosome_levels,
        ordered = TRUE
      )
    ]

    data.table::setorder(
      gwas_data,
      CHROM,
      GENPOS
    )


    # Calculate cumulative genomic positions

    chromosome_information <- gwas_data[
      ,
      .(
        chromosome_length = max(
          GENPOS,
          na.rm = TRUE
        )
      ),
      by = CHROM
    ]

    data.table::setorder(
      chromosome_information,
      CHROM
    )

    chromosome_information[
      ,
      chromosome_start := data.table::shift(
        cumsum(chromosome_length),
        fill = 0
      )
    ]

    chromosome_information[
      ,
      chromosome_midpoint :=
        chromosome_start +
        chromosome_length / 2
    ]

    gwas_data[
      chromosome_information,
      on = "CHROM",
      chromosome_start := i.chromosome_start
    ]

    gwas_data[
      ,
      cumulative_position :=
        GENPOS + chromosome_start
    ]

    gwas_data[
      ,
      chromosome_colour :=
        as.integer(CHROM) %% 2L
    ]


    # Identify significant variants

    significant_variants <- gwas_data[
      LOG10P >= log10p_threshold
    ]

    message(
      "Genome-wide significant variants for ",
      group,
      ": ",
      format(
        nrow(significant_variants),
        big.mark = ","
      )
    )


    # Select variants to label

    if (
      nrow(significant_variants) == 0L ||
      maximum_labels == 0L
    ) {

      labelled_variants <- significant_variants[0]

    } else if (label_mode == "variant") {

      # Label the strongest individual variants
      labelled_variants <- significant_variants[
        order(-LOG10P)
      ][
        seq_len(
          min(.N, maximum_labels)
        )
      ]

    } else {

      # Divide each chromosome into windows and label the
      # strongest significant variant per window
      labelled_variants <- data.table::copy(
        significant_variants
      )

      labelled_variants[
        ,
        label_window :=
          floor(GENPOS / label_window_bp)
      ]

      labelled_variants <- labelled_variants[
        order(-LOG10P)
      ][
        ,
        .SD[1L],
        by = .(
          CHROM,
          label_window
        )
      ][
        order(-LOG10P)
      ]

      labelled_variants <- labelled_variants[
        seq_len(
          min(.N, maximum_labels)
        )
      ]
    }

    labelled_variants[
      is.na(ID) |
        !nzchar(trimws(ID)) |
        ID == ".",
      ID := paste0(
        "chr",
        as.character(CHROM),
        ":",
        format(
          GENPOS,
          scientific = FALSE,
          trim = TRUE
        )
      )
    ]

    message(
      "Variants labelled for ",
      group,
      ": ",
      nrow(labelled_variants)
    )


    # Create the Manhattan plot

    manhattan_plot <- ggplot2::ggplot(
      gwas_data,
      ggplot2::aes(
        x = cumulative_position,
        y = LOG10P,
        colour = factor(chromosome_colour)
      )
    ) +
      ggplot2::geom_point(
        alpha = 0.65,
        size = 0.7
      ) +
      ggplot2::geom_hline(
        yintercept = log10p_threshold,
        colour = "#B2182B",
        linetype = "dashed",
        linewidth = 0.7
      ) +
      ggplot2::scale_colour_manual(
        values = c(
          "0" = "#2166AC",
          "1" = "#67A9CF"
        )
      ) +
      ggplot2::scale_x_continuous(
        breaks =
          chromosome_information$chromosome_midpoint,
        labels =
          as.character(chromosome_information$CHROM),
        expand = ggplot2::expansion(
          mult = c(0.01, 0.01)
        )
      ) +
      ggplot2::scale_y_continuous(
        expand = ggplot2::expansion(
          mult = c(0.02, 0.12)
        )
      ) +
      ggplot2::labs(
        title = paste0(
          phenotype_name,
          ": ",
          group,
          " Manhattan plot"
        ),
        subtitle = paste0(
          "Genome-wide significance threshold: P < ",
          format(
            gws_threshold,
            scientific = TRUE
          )
        ),
        x = "Chromosome",
        y = expression(-logitalic(P))
      ) +
      ggplot2::theme_classic(
        base_size = 12
      ) +
      ggplot2::theme(
        legend.position = "none",
        axis.text.x = ggplot2::element_text(
          size = 9,
          colour = "black"
        ),
        axis.text.y = ggplot2::element_text(
          colour = "black"
        ),
        axis.line.x = ggplot2::element_blank(),
        panel.grid.major.y = ggplot2::element_line(
          colour = "grey90",
          linewidth = 0.3
        ),
        plot.title = ggplot2::element_text(
          face = "bold",
          size = 16
        ),
        plot.subtitle = ggplot2::element_text(
          size = 10
        )
      )

    # Add highlights and labels only when labels exist

    if (nrow(labelled_variants) > 0L) {

      manhattan_plot <- manhattan_plot +
        ggplot2::geom_point(
          data = labelled_variants,
          mapping = ggplot2::aes(
            x = cumulative_position,
            y = LOG10P
          ),
          inherit.aes = FALSE,
          colour = "#B2182B",
          size = 2
        ) +
        ggrepel::geom_label_repel(
          data = labelled_variants,
          mapping = ggplot2::aes(
            x = cumulative_position,
            y = LOG10P,
            label = ID
          ),
          inherit.aes = FALSE,
          size = 3,
          colour = "black",
          fill = "white",
          box.padding = 0.4,
          point.padding = 0.3,
          min.segment.length = 0,
          segment.colour = "grey50",
          segment.linewidth = 0.3,
          max.overlaps = Inf,
          seed = 123,
          show.legend = FALSE
        )
    }


    # Store outputs

    gwas_results[[group]] <- gwas_data
    significant_results[[group]] <- significant_variants
    labelled_results[[group]] <- labelled_variants
    manhattan_plots[[group]] <- manhattan_plot

    # Optionally save the plot

    if (isTRUE(save_plots)) {

      plot_file <- file.path(
        output_directory,
        paste0(
          phenotype_name,
          "_",
          group,
          "_manhattan_labelled.png"
        )
      )

      ggplot2::ggsave(
        filename = plot_file,
        plot = manhattan_plot,
        width = plot_width,
        height = plot_height,
        units = "in",
        dpi = plot_dpi,
        bg = "white"
      )

      message(
        "Plot saved to: ",
        plot_file
      )
    }


    # Remove the local file if it is no longer needed

    if (!isTRUE(keep_downloaded_files)) {
      unlink(local_file)
      download_paths[[group]] <- NULL
    }
  }

  # Confirm that at least one plot was produced
  if (length(manhattan_plots) == 0L) {
    stop(
      paste0(
        "No Manhattan plots were produced. ",
        "Review the preceding warnings and messages."
      )
    )
  }

  output <- list(
    plots = manhattan_plots,
    gwas_results = gwas_results,
    significant_results = significant_results,
    labelled_results = labelled_results,
    file_information = file_information,
    downloaded_files = download_paths,
    settings = list(
      phenotype_name = phenotype_name,
      gwas_directory = gwas_directory,
      requested_groups = genetic_similarity_groups,
      processed_groups = names(manhattan_plots),
      gws_threshold = gws_threshold,
      log10p_threshold = log10p_threshold,
      maximum_labels = maximum_labels,
      label_mode = label_mode,
      label_window_bp = label_window_bp
    )
  )

  message("")
  message(
    "Successfully produced Manhattan plots for: ",
    paste(
      names(manhattan_plots),
      collapse = ", "
    )
  )

  return(output)
}



########################################
#===== Function 2: Variant search =====#
########################################

find_variant <- function(
    phenotype_name,
    gwas_directory,
    rsid = NULL,
    chr_pos_ref_alt = NULL,
    genetic_similarity_groups = c(
      "eur", "afr", "eas", "cas", "sas", "mid", "amr"
    )
) {

  # Require at least one identifier
  if (is.null(rsid) && is.null(chr_pos_ref_alt)) {
    stop(
      paste0(
        "Supply an `rsid` and/or `chr_pos_ref_alt` ",
        "in CHR:POS:REF:ALT format."
      )
    )
  }

  awk_conditions <- character()


  # Match rsID against the ID column

  if (!is.null(rsid)) {

    rsid <- trimws(as.character(rsid))

    if (
      length(rsid) != 1L ||
      is.na(rsid) ||
      !nzchar(rsid)
    ) {
      stop("`rsid` must be a single non-empty character string.")
    }

    # REGENIE columns:
    # $1 = CHROM
    # $2 = GENPOS
    # $3 = ID
    # $4 = ALLELE0
    # $5 = ALLELE1

    awk_conditions <- c(
      awk_conditions,
      sprintf(
        'tolower($3) == tolower("%s")',
        rsid
      )
    )
  }


  # Dissect and match CHR:POS:REF:ALT

  if (!is.null(chr_pos_ref_alt)) {

    chr_pos_ref_alt <- trimws(
      as.character(chr_pos_ref_alt)
    )

    if (
      length(chr_pos_ref_alt) != 1L ||
      is.na(chr_pos_ref_alt) ||
      !nzchar(chr_pos_ref_alt)
    ) {
      stop(
        paste0(
          "`chr_pos_ref_alt` must be a single non-empty ",
          "character string."
        )
      )
    }

    variant_components <- strsplit(
      chr_pos_ref_alt,
      split = ":",
      fixed = TRUE
    )[[1L]]

    if (length(variant_components) != 4L) {
      stop(
        paste0(
          "`chr_pos_ref_alt` must use CHR:POS:REF:ALT format, ",
          "for example `8:127735434:C:T`."
        )
      )
    }

    chromosome <- toupper(
      trimws(variant_components[1L])
    )

    # Allow either "8" or "chr8" as input
    chromosome <- sub(
      "^CHR",
      "",
      chromosome
    )

    position <- trimws(
      variant_components[2L]
    )

    allele0 <- toupper(
      trimws(variant_components[3L])
    )

    allele1 <- toupper(
      trimws(variant_components[4L])
    )

    # Validate position
    if (!grepl("^[0-9]+$", position)) {
      stop(
        "The POS component of `chr_pos_ref_alt` must be an integer."
      )
    }

    # Validate chromosome and alleles
    if (
      !nzchar(chromosome) ||
      !nzchar(allele0) ||
      !nzchar(allele1)
    ) {
      stop(
        paste0(
          "The CHR, REF, and ALT components of ",
          "`chr_pos_ref_alt` must not be empty."
        )
      )
    }

    # Match:
    # $1 = CHROM
    # $2 = GENPOS
    # $4 = ALLELE0
    # $5 = ALLELE1
    #
    # The chromosome condition supports both 8 and chr8
    # in the REGENIE file.

    positional_condition <- sprintf(
      paste0(
        '((toupper($1) == "%s" || ',
        'toupper($1) == "CHR%s") && ',
        '$2 == "%s" && ',
        'toupper($4) == "%s" && ',
        'toupper($5) == "%s")'
      ),
      chromosome,
      chromosome,
      position,
      allele0,
      allele1
    )

    awk_conditions <- c(
      awk_conditions,
      positional_condition
    )
  }


  # Combine supplied searches using OR

  awk_condition <- paste(
    awk_conditions,
    collapse = " || "
  )


  # Search each genetic similarity group

  all_matches <- list()

  for (group in genetic_similarity_groups) {

    remote_file <- paste0(
      sub("/+$", "", gwas_directory),
      "/",
      phenotype_name,
      "_",
      group,
      ".regenie"
    )

    message(
      "\nSearching group: ",
      toupper(group)
    )

    shell_command <- paste0(
      "unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy; ",
      "dx cat ",
      shQuote(remote_file, type = "sh"),
      " | awk 'NR == 1 || ",
      awk_condition,
      "'"
    )

    output <- tryCatch(
      system(
        paste(
          "bash -c",
          shQuote(
            shell_command,
            type = "sh"
          )
        ),
        intern = TRUE,
        ignore.stderr = TRUE
      ),
      warning = function(w) {
        character()
      },
      error = function(e) {
        character()
      }
    )


    # Print the result or a not-found message

    # When no variant is found, output contains only the header
    if (length(output) <= 1L) {

      message(
        "Variant not found in group: ",
        toupper(group)
      )

      next
    }

    matched_rows <- data.table::fread(
      text = paste(
        output,
        collapse = "\n"
      ),
      header = TRUE
    )

    if ("#CHROM" %in% names(matched_rows)) {
      data.table::setnames(
        matched_rows,
        old = "#CHROM",
        new = "CHROM"
      )
    }

    matched_rows[
      ,
      GENETIC_SIMILARITY_GROUP := toupper(group)
    ]

    data.table::setcolorder(
      matched_rows,
      c(
        "GENETIC_SIMILARITY_GROUP",
        setdiff(
          names(matched_rows),
          "GENETIC_SIMILARITY_GROUP"
        )
      )
    )

    all_matches[[toupper(group)]] <- matched_rows
  }

  # Combine matches from all groups
  if (length(all_matches) == 0L) {
    message("\nVariant was not found in any requested group.")
    return(invisible(NULL))
  }

  combined_matches <- data.table::rbindlist(
    all_matches,
    use.names = TRUE,
    fill = TRUE
  )


  # Create a raw p-value where LOG10P is available
  if ("LOG10P" %in% names(combined_matches)) {
    combined_matches[
      ,
      P := 10^(-LOG10P)
    ]
  }


  # Select the most informative REGENIE columns for display
  preferred_columns <- c(
    "GENETIC_SIMILARITY_GROUP",
    "CHROM",
    "GENPOS",
    "ID",
    "ALLELE0",
    "ALLELE1",
    "A1FREQ",
    "N",
    "TEST",
    "BETA",
    "SE",
    "CHISQ",
    "LOG10P",
    "P"
  )

  display_columns <- intersect(
    preferred_columns,
    names(combined_matches)
  )

  display_table <- data.table::copy(
    combined_matches[
      ,
      ..display_columns
    ]
  )


  # Format numeric results for readability
  if ("A1FREQ" %in% names(display_table)) {
    display_table[
      ,
      A1FREQ := formatC(
        A1FREQ,
        format = "f",
        digits = 4
      )
    ]
  }

  if ("BETA" %in% names(display_table)) {
    display_table[
      ,
      BETA := formatC(
        BETA,
        format = "f",
        digits = 4
      )
    ]
  }

  if ("SE" %in% names(display_table)) {
    display_table[
      ,
      SE := formatC(
        SE,
        format = "f",
        digits = 4
      )
    ]
  }

  if ("CHISQ" %in% names(display_table)) {
    display_table[
      ,
      CHISQ := formatC(
        CHISQ,
        format = "f",
        digits = 3
      )
    ]
  }

  if ("LOG10P" %in% names(display_table)) {
    display_table[
      ,
      LOG10P := formatC(
        LOG10P,
        format = "f",
        digits = 3
      )
    ]
  }

  if ("P" %in% names(display_table)) {
    display_table[
      ,
      P := formatC(
        P,
        format = "e",
        digits = 3
      )
    ]
  }


  # Use more readable column headings
  display_names <- c(
    GENETIC_SIMILARITY_GROUP = "Group",
    CHROM = "Chromosome",
    GENPOS = "Position",
    ID = "Variant ID",
    ALLELE0 = "Allele 0",
    ALLELE1 = "Allele 1",
    A1FREQ = "Allele 1 frequency",
    N = "N",
    TEST = "Test",
    BETA = "Beta",
    SE = "SE",
    CHISQ = "Chi-square",
    LOG10P = "-log10(P)",
    P = "P-value"
  )

  data.table::setnames(
    display_table,
    old = names(display_table),
    new = unname(
      display_names[names(display_table)]
    )
  )


  # Create the formatted HTML table
  html_table <- knitr::kable(
    display_table,
    format = "html",
    row.names = FALSE,
    caption = paste0(
      "REGENIE results for the variant of interest"
    ),
    align = "c",
    escape = TRUE
  )

  # Display properly in Jupyter where IRdisplay is available
  if (requireNamespace("IRdisplay", quietly = TRUE)) {
    IRdisplay::display_html(
      as.character(html_table)
    )
  } else {
    print(html_table)
  }


  # Return the original, unformatted results invisibly
  invisible(combined_matches)
}




########################################
#==== Function 3: Display top hits ====#
########################################

display_top_hits <- function(
    phenotype_name,
    gwas_directory,
    genetic_similarity_groups = c(
      "eur", "afr", "eas", "cas", "sas", "mid", "amr"
    ),
    n_top_hits = 10L,
    metric = c("LOG10P", "BETA", "CHISQ"),
    beta_sort = c("absolute", "positive", "negative")
) {

  # Check and standardise arguments

  beta_sort <- match.arg(beta_sort)

  genetic_similarity_groups <- unique(
    tolower(genetic_similarity_groups)
  )

  n_top_hits <- as.integer(n_top_hits)

  if (
    length(n_top_hits) != 1L ||
    is.na(n_top_hits) ||
    n_top_hits < 1L
  ) {
    stop("`n_top_hits` must be a positive integer.")
  }

  allowed_metrics <- c(
    "LOG10P",
    "BETA",
    "CHISQ"
  )

  metric <- unique(
    toupper(metric)
  )

  invalid_metrics <- setdiff(
    metric,
    allowed_metrics
  )

  if (length(invalid_metrics) > 0L) {
    stop(
      paste0(
        "Invalid metric(s): ",
        paste(invalid_metrics, collapse = ", "),
        ". Valid metrics are LOG10P, BETA, and CHISQ."
      )
    )
  }

  if (!nzchar(Sys.which("dx"))) {
    stop(
      paste0(
        "The DNAnexus `dx` command-line tool could not be found. ",
        "This function must be run where `dx` is available and ",
        "authenticated."
      )
    )
  }


  # Prepare one result list per requested metric

  top_hits <- setNames(
    vector(
      mode = "list",
      length = length(metric)
    ),
    metric
  )

  for (metric_name in metric) {
    top_hits[[metric_name]] <- list()
  }

  # Process each genetic similarity group

  for (group in genetic_similarity_groups) {

    group_upper <- toupper(group)

    remote_file <- paste0(
      sub("/+$", "", gwas_directory),
      "/",
      phenotype_name,
      "_",
      group,
      ".regenie"
    )

    message(
      "\nSearching group: ",
      group_upper
    )

    # Read the header only

    header_command <- paste0(
      "unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy; ",
      "dx cat ",
      shQuote(remote_file, type = "sh"),
      " | head -n 1"
    )

    header_output <- tryCatch(
      system(
        paste(
          "bash -c",
          shQuote(
            header_command,
            type = "sh"
          )
        ),
        intern = TRUE,
        ignore.stderr = TRUE
      ),
      warning = function(w) {
        character()
      },
      error = function(e) {
        character()
      }
    )

    if (length(header_output) == 0L) {
      message(
        "REGENIE file not found or could not be read for group: ",
        group_upper
      )

      next
    }

    header <- strsplit(
      trimws(header_output[1L]),
      split = "[[:space:]]+"
    )[[1L]]

    # Standardise #CHROM to CHROM
    header[header == "#CHROM"] <- "CHROM"

    # Check the standard display columns

    display_columns <- c(
      "CHROM",
      "GENPOS",
      "ID",
      "ALLELE0",
      "ALLELE1",
      "A1FREQ",
      "N",
      "TEST",
      "BETA",
      "SE",
      "CHISQ",
      "LOG10P"
    )

    available_display_columns <- intersect(
      display_columns,
      header
    )

    display_indices <- match(
      available_display_columns,
      header
    )

    # Produce top-hit results for each requested metric

    for (metric_name in metric) {

      metric_index <- match(
        metric_name,
        header
      )

      if (is.na(metric_index)) {
        message(
          metric_name,
          " was not found in the REGENIE file for group: ",
          group_upper
        )

        next
      }

      # Create the awk ranking expression

      if (
        metric_name == "BETA" &&
        beta_sort == "absolute"
      ) {

        # Rank by absolute beta while preserving original beta
        ranking_expression <- paste0(
          "value = $",
          metric_index,
          "; ",
          "if (value < 0) value = -value; "
        )

      } else if (
        metric_name == "BETA" &&
        beta_sort == "negative"
      ) {

        # Multiply by -1 so the most negative beta ranks highest
        ranking_expression <- paste0(
          "value = -$",
          metric_index,
          "; "
        )

      } else {

        # Largest LOG10P, CHISQ, or positive BETA first
        ranking_expression <- paste0(
          "value = $",
          metric_index,
          "; "
        )
      }

      # Stream, rank and retain only the top rows

      awk_print_columns <- paste0(
        "$",
        display_indices,
        collapse = ", "
      )

      top_hits_command <- paste0(
        "unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy; ",
        "dx cat ",
        shQuote(remote_file, type = "sh"),
        " | awk '",
        "BEGIN {OFS=\"\\t\"} ",
        "NR > 1 {",
        ranking_expression,
        "if ($",
        metric_index,
        " != \"\" && $",
        metric_index,
        " != \"NA\") ",
        "print value, ",
        awk_print_columns,
        "}' ",
        "| sort -t $'\\t' -k1,1gr ",
        "| head -n ",
        n_top_hits,
        " | cut -f2-"
      )

      top_hits_output <- tryCatch(
        system(
          paste(
            "bash -c",
            shQuote(
              top_hits_command,
              type = "sh"
            )
          ),
          intern = TRUE,
          ignore.stderr = TRUE
        ),
        warning = function(w) {
          character()
        },
        error = function(e) {
          character()
        }
      )

      if (length(top_hits_output) == 0L) {
        message(
          "No valid ",
          metric_name,
          " results found for group: ",
          group_upper
        )

        next
      }

      # Import only the selected top rows into R

      table_text <- paste(
        c(
          paste(
            available_display_columns,
            collapse = "\t"
          ),
          top_hits_output
        ),
        collapse = "\n"
      )

      group_top_hits <- data.table::fread(
        text = table_text,
        header = TRUE,
        sep = "\t"
      )

      group_top_hits[
        ,
        GENETIC_SIMILARITY_GROUP := group_upper
      ]

      data.table::setcolorder(
        group_top_hits,
        c(
          "GENETIC_SIMILARITY_GROUP",
          setdiff(
            names(group_top_hits),
            "GENETIC_SIMILARITY_GROUP"
          )
        )
      )

      top_hits[[metric_name]][[group_upper]] <- group_top_hits

      message(
        "Found ",
        nrow(group_top_hits),
        " top hits by ",
        metric_name,
        " for ",
        group_upper,
        "."
      )
    }
  }

  # Combine and display one table per requested metric

  combined_results <- list()

  for (metric_name in metric) {

    if (length(top_hits[[metric_name]]) == 0L) {
      message(
        "\nNo results were available for metric: ",
        metric_name
      )

      next
    }

    metric_results <- data.table::rbindlist(
      top_hits[[metric_name]],
      use.names = TRUE,
      fill = TRUE
    )

    # Add ranks within each group

    if (
      metric_name == "BETA" &&
      beta_sort == "absolute"
    ) {

      metric_results[
        ,
        RANK := data.table::frank(
          -abs(BETA),
          ties.method = "first"
        ),
        by = GENETIC_SIMILARITY_GROUP
      ]

      data.table::setorder(
        metric_results,
        GENETIC_SIMILARITY_GROUP,
        RANK
      )

    } else if (
      metric_name == "BETA" &&
      beta_sort == "negative"
    ) {

      metric_results[
        ,
        RANK := data.table::frank(
          BETA,
          ties.method = "first"
        ),
        by = GENETIC_SIMILARITY_GROUP
      ]

      data.table::setorder(
        metric_results,
        GENETIC_SIMILARITY_GROUP,
        RANK
      )

    } else {

      metric_results[
        ,
        RANK := data.table::frank(
          -get(metric_name),
          ties.method = "first"
        ),
        by = GENETIC_SIMILARITY_GROUP
      ]

      data.table::setorder(
        metric_results,
        GENETIC_SIMILARITY_GROUP,
        RANK
      )
    }


    # Calculate raw p-values
    if ("LOG10P" %in% names(metric_results)) {
      metric_results[
        ,
        P := 10^(-LOG10P)
      ]
    }

    combined_results[[metric_name]] <- metric_results

    # Create a formatted copy for HTML display

    preferred_columns <- c(
      "GENETIC_SIMILARITY_GROUP",
      "RANK",
      "CHROM",
      "GENPOS",
      "ID",
      "ALLELE0",
      "ALLELE1",
      "A1FREQ",
      "N",
      "TEST",
      "BETA",
      "SE",
      "CHISQ",
      "LOG10P",
      "P"
    )

    columns_to_display <- intersect(
      preferred_columns,
      names(metric_results)
    )

    display_table <- data.table::copy(
      metric_results[
        ,
        ..columns_to_display
      ]
    )

    # Format numeric values

    four_decimal_columns <- intersect(
      c(
        "A1FREQ",
        "BETA",
        "SE"
      ),
      names(display_table)
    )

    for (column in four_decimal_columns) {
      data.table::set(
        display_table,
        j = column,
        value = formatC(
          display_table[[column]],
          format = "f",
          digits = 4
        )
      )
    }

    if ("CHISQ" %in% names(display_table)) {
      display_table[
        ,
        CHISQ := formatC(
          CHISQ,
          format = "f",
          digits = 3
        )
      ]
    }

    if ("LOG10P" %in% names(display_table)) {
      display_table[
        ,
        LOG10P := formatC(
          LOG10P,
          format = "f",
          digits = 3
        )
      ]
    }

    if ("P" %in% names(display_table)) {
      display_table[
        ,
        P := formatC(
          P,
          format = "e",
          digits = 3
        )
      ]
    }

    # Rename columns for display

    display_names <- c(
      GENETIC_SIMILARITY_GROUP = "Group",
      RANK = "Rank",
      CHROM = "Chromosome",
      GENPOS = "Position",
      ID = "Variant ID",
      ALLELE0 = "Allele 0",
      ALLELE1 = "Allele 1",
      A1FREQ = "Allele 1 frequency",
      N = "N",
      TEST = "Test",
      BETA = "Beta",
      SE = "SE",
      CHISQ = "Chi-square",
      LOG10P = "-log10(P)",
      P = "P-value"
    )

    data.table::setnames(
      display_table,
      old = names(display_table),
      new = unname(
        display_names[names(display_table)]
      )
    )

    # Create the table caption

    metric_description <- switch(
      metric_name,
      LOG10P = "-log10(P)",
      CHISQ = "chi-squared statistic",
      BETA = switch(
        beta_sort,
        absolute = "absolute beta",
        positive = "positive beta",
        negative = "negative beta"
      )
    )

    table_caption <- paste0(
      "Top ",
      n_top_hits,
      " REGENIE hits per genetic similarity group by ",
      metric_description
    )


    # Create and display the formatted HTML table

    html_table <- knitr::kable(
      display_table,
      format = "html",
      row.names = FALSE,
      caption = table_caption,
      align = "c",
      escape = TRUE
    )

    if (requireNamespace("IRdisplay", quietly = TRUE)) {
      IRdisplay::display_html(
        as.character(html_table)
      )
    } else {
      print(html_table)
    }
  }

  # Check that at least one result table was produced

  if (length(combined_results) == 0L) {
    message(
      "\nNo top-hit tables could be produced."
    )

    return(
      invisible(NULL)
    )
  }

  # Report completion

  message(
    "\nSuccessfully produced top-hit tables for: ",
    paste(
      names(combined_results),
      collapse = ", "
    )
  )

  # Return original numeric results
  invisible(combined_results)
}

## Applying the functions

In [ ]:
manhattan_results <- plot_manhattan(
    phenotype_name = "PRCA",
    gwas_directory = "project-J8xyP9k2FFf520KYVzfvQ5J5:/GWAS/PrCa_o40/merged",
    genetic_similarity_groups = c("eur", "afr"),
    gws_threshold = 5e-8,
    maximum_labels = 30,
    label_mode = "locus",
    label_window_bp = 500000,
    save_plots = TRUE,
    output_directory = getwd()
)

In [ ]:
find_variant(
    phenotype_name = "PRCA",
    gwas_directory = "project-J8xyP9k2FFf520KYVzfvQ5J5:/GWAS/PrCa_o40/merged",
    genetic_similarity_groups = c("eur", "afr"),
    rsid = "rs72725854",
    chr_pos_ref_alt = "8:127062570:A:T"
)

In [ ]:
display_top_hits(
    phenotype_name = "PRCA",
    gwas_directory = "project-J8xyP9k2FFf520KYVzfvQ5J5:/GWAS/PrCa_o40/merged",
    genetic_similarity_groups = c("eur", "afr"),
    n_top_hits = 10L,
    metric = c("BETA", "LOG10P"),
    beta_sort = "absolute"
)